In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

from src.schema import DATE_COL, STORE_COL, ITEM_COL, SALES_COL
from src.backtest import rolling_origin_windows, evaluate_forecast, run_backtest
from src.metrics import wape, rmse, mape, bias


In [ ]:
# Load and prepare training data
df_raw = pd.read_csv('../data/raw/train.csv')
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])

df = df_raw.sort_values(
    [STORE_COL, ITEM_COL, DATE_COL]
).reset_index(drop=True)


In [ ]:
# Generate rolling validation windows
windows = rolling_origin_windows(df, horizon=28, n_windows=3)


In [ ]:
# Generate classical and baseline forecasts independently for each rolling window

from src.baselines import forecast_by_group

classical_bt = pd.read_csv("../data/processed/forecasts_classical_backtest.csv")
classical_bt[DATE_COL] = pd.to_datetime(classical_bt[DATE_COL])

top10_keys = classical_bt[[STORE_COL, ITEM_COL]].drop_duplicates()

assert len(top10_keys) == 10, (
    f"Expected 10 store-item pairs, found {len(top10_keys)}"
)

print("Classical forecasts loaded:", classical_bt.shape)
print("Common store-item pairs:", len(top10_keys))


def combined_forecast(train_df, val_df):
    # Validation dates for this rolling window
    val_dates = pd.DatetimeIndex(sorted(val_df[DATE_COL].unique()))
    horizon = len(val_dates)

    # Select Prophet and SARIMA forecasts for this window
    classical_window = classical_bt[
        classical_bt[DATE_COL].isin(val_dates)
    ].copy()

    assert len(classical_window) == 560, (
        f"Expected 560 classical rows, got {len(classical_window)}"
    )

    # Restrict training data to the same 10 store-item pairs
    train_top10 = train_df.merge(
        top10_keys,
        on=[STORE_COL, ITEM_COL],
        how="inner"
    )

    # Generate baselines using ONLY this window's training data
    baseline_specs = [
        ("naive", {}),
        ("seasonal_naive", {"season": 7}),
        ("moving_average", {"window": 7}),
        ("moving_average", {"window": 28}),
    ]

    baseline_forecasts = []

    for method, kwargs in baseline_specs:
        forecast = forecast_by_group(
            train_top10,
            method=method,
            horizon=horizon,
            **kwargs
        )

        assert len(forecast) == 280, (
            f"{method} {kwargs}: expected 280 rows, got {len(forecast)}"
        )

        assert set(pd.to_datetime(forecast[DATE_COL]).unique()) == set(val_dates), (
            f"{method} {kwargs}: forecast dates do not match validation dates"
        )

        baseline_forecasts.append(forecast)

    # Combine Prophet, SARIMA and all four baselines
    combined = pd.concat(
        [classical_window] + baseline_forecasts,
        ignore_index=True
    )

    combined[DATE_COL] = pd.to_datetime(combined[DATE_COL])

    return combined[
        [DATE_COL, STORE_COL, ITEM_COL, "y_hat", "model"]
    ]


# Run all three rolling windows
comparison_table, detailed = run_backtest(
    combined_forecast,
    df,
    n_windows=3,
    horizon=28
)

print("\nComparison across all three windows:")
print(comparison_table.to_string(index=False))

# Verify every model was evaluated on all 840 rows
assert set(comparison_table["n_rows"]) == {840}, (
    "Not all models were evaluated on 840 rows"
)

assert set(comparison_table["n_windows"]) == {3}, (
    "Not all models were evaluated across 3 windows"
)


In [ ]:
# Save backtesting results
comparison_columns = ['model', 'wape', 'rmse', 'mape', 'bias', 'rank']

comparison_table[comparison_columns].to_csv(
    '../data/processed/model_comparison_shryes.csv',
    index=False
)
